# VAE x64 Latent Dimension Sweep

This notebook is the curated submission-facing entry point for the VAE latent dimension sweep on the shared x64 anomaly benchmark.
It defaults to reusing the saved per-latent-dim artifacts instead of rerunning the sweep.


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Sweep config: `experiments/anomaly_detection/vae/x64/latent_dim_sweep/train_config.toml`
- Training script: `scripts/train_vae.py`
- Artifact root: `experiments/anomaly_detection/vae/x64/latent_dim_sweep/artifacts/vae_latent_dim_sweep/`
- Default behavior: reuse the saved `results/latent_dim_sweep_summary.json` and the per-latent-dim checkpoints and evaluation outputs


### Setup And Imports

This cell resolves the repo root, exposes `src/` on `PYTHONPATH`, and imports the shared config loader plus the standard utilities used to read the saved sweep artifacts.


In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:

    def display(obj: object) -> None:
        print(obj)

def warn_skip(message: str) -> None:
    print(f'[WARNING] {message}')

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'wafer_defect').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml


### Load The Sweep Config And Run Controls

This cell points the notebook at the local latent-dim-sweep config and exposes the rerun flags. Leave them as `False` to keep the notebook in artifact-first mode.


In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'latent_dim_sweep' / 'train_config.toml'
config = load_toml(CONFIG_PATH)
SWEEP_ROOT = REPO_ROOT / config['run']['output_dir']
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR = SWEEP_ROOT / 'configs'
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = SWEEP_ROOT / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RETRAIN = False
display(config)


### Optionally Rerun The Sweep

This cell is only used when you explicitly enable the sweep rerun. Otherwise it simply reuses the saved summary and per-latent-dim artifacts already present in the repository.


In [ ]:
def format_toml_value(value):
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, (int, float)):
        return repr(value)
    escaped = str(value).replace('\\', '\\\\').replace('"', '\\"')
    return f'"{escaped}"'

def dump_toml(config_dict):
    lines = []
    for section, values in config_dict.items():
        lines.append(f'[{section}]')
        for key, value in values.items():
            lines.append(f'{key} = {format_toml_value(value)}')
        lines.append('')
    return '\n'.join(lines).rstrip() + '\n'

def latent_dim_tag(latent_dim_value: int) -> str:
    return f'latent_dim_{int(latent_dim_value)}'
summary_path = SWEEP_ROOT / 'results' / 'latent_dim_sweep_summary.json'
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC_ROOT) if not env.get('PYTHONPATH') else str(SRC_ROOT) + os.pathsep + env['PYTHONPATH']
if RETRAIN:
    latent_dims = list(config['sweep']['latent_dims'])
    results = []
    for latent_dim_value in latent_dims:
        tag = latent_dim_tag(latent_dim_value)
        run_output_dir = SWEEP_ROOT / tag
        run_config = {'run': {'output_dir': run_output_dir.relative_to(REPO_ROOT).as_posix(), 'seed': config['run']['seed']}, 'data': dict(config['data']), 'training': dict(config['training']), 'model': {'type': 'vae', 'latent_dim': int(latent_dim_value), 'beta': 0.005}}
        run_config_path = CONFIGS_DIR / f'train_vae_{tag}.toml'
        run_config_path.write_text(dump_toml(run_config), encoding='utf-8')
        train_cmd = [sys.executable, 'scripts/train_vae.py', '--config', str(run_config_path.relative_to(REPO_ROOT))]
        eval_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str((run_output_dir / 'checkpoints' / 'best_model.pt').relative_to(REPO_ROOT)), '--config', str(run_config_path.relative_to(REPO_ROOT)), '--output-dir', str((run_output_dir / 'evaluation').relative_to(REPO_ROOT))]
        subprocess.run(train_cmd, cwd=REPO_ROOT, env=env, check=True)
        subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, check=True)
        run_summary = json.loads((run_output_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))
        results.append({'latent_dim': int(latent_dim_value), 'threshold': float(run_summary['threshold']), 'val_threshold_precision': float(run_summary['metrics_at_validation_threshold']['precision']), 'val_threshold_recall': float(run_summary['metrics_at_validation_threshold']['recall']), 'val_threshold_f1': float(run_summary['metrics_at_validation_threshold']['f1']), 'auroc': float(run_summary['metrics_at_validation_threshold']['auroc']), 'auprc': float(run_summary['metrics_at_validation_threshold']['auprc']), 'best_sweep_threshold': float(run_summary['best_threshold_sweep']['threshold']), 'best_sweep_precision': float(run_summary['best_threshold_sweep']['precision']), 'best_sweep_recall': float(run_summary['best_threshold_sweep']['recall']), 'best_sweep_f1': float(run_summary['best_threshold_sweep']['f1'])})
    summary_path.write_text(json.dumps({'latent_dims': latent_dims, 'results': results}, indent=2), encoding='utf-8')
elif summary_path.exists():
    print(f'Found existing latent-dim sweep artifacts in {SWEEP_ROOT}. Skipping rerun.')
else:
    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')


### Load The Saved Sweep Summary

This cell reads the aggregated summary and builds a dataframe that we can use for the comparison plots and tables.


In [ ]:
sweep_ready = False
summary_payload = {}
latent_dim_sweep_df = pd.DataFrame()
if summary_path.exists():
    summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
    latent_dim_sweep_df = pd.DataFrame(summary_payload['results']).sort_values('val_threshold_f1', ascending=False).reset_index(drop=True)
    sweep_ready = not latent_dim_sweep_df.empty
    display(latent_dim_sweep_df)
else:
    warn_skip('Saved sweep artifacts are missing, so the cached sweep review cells will be skipped.')


### Plot Sweep Metrics Across `beta`

This cell recreates the main sweep comparison figure, shows it inline, and saves it to the local `plots/` folder.


In [ ]:
if not sweep_ready:
    warn_skip('Sweep metric plots are unavailable because the saved sweep summary is missing.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    sorted_df = latent_dim_sweep_df.sort_values('latent_dim').reset_index(drop=True)
    axes[0].plot(sorted_df['latent_dim'], sorted_df['val_threshold_f1'], marker='o', label='val-threshold F1')
    axes[0].plot(sorted_df['latent_dim'], sorted_df['best_sweep_f1'], marker='s', label='best-sweep F1')
    axes[0].set_title('F1 vs latent-dim')
    axes[0].set_xlabel('latent_dim')
    axes[0].legend()
    axes[1].plot(sorted_df['latent_dim'], sorted_df['auroc'], marker='o', color='#1f77b4')
    axes[1].set_title('AUROC vs latent-dim')
    axes[1].set_xlabel('latent_dim')
    axes[2].plot(sorted_df['latent_dim'], sorted_df['auprc'], marker='o', color='#2ca02c')
    axes[2].set_title('AUPRC vs latent-dim')
    axes[2].set_xlabel('latent_dim')
    fig.savefig(PLOTS_DIR / 'latent_dim_sweep_metrics.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)


### Compare Training Curves Across Betas

This cell loads each saved `history.json`, overlays the validation curves, and saves the comparison plot for the report.


In [ ]:
if not sweep_ready:
    warn_skip('Training-curve comparison is unavailable because the saved sweep summary is missing.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    plotted_any = False
    for sweep_value in sorted(latent_dim_sweep_df['latent_dim'].tolist()):
        tag = latent_dim_tag(sweep_value)
        history_path = SWEEP_ROOT / tag / 'results' / 'history.json'
        if not history_path.exists():
            warn_skip(f'Missing training history for {tag}: {history_path}. Skipping this variant in the plot.')
            continue
        history_df = pd.DataFrame(json.loads(history_path.read_text(encoding='utf-8')))
        axes[0].plot(history_df['epoch'], history_df['val_loss'], label=f'latent_dim={sweep_value}')
        axes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label=f'latent_dim={sweep_value}')
        plotted_any = True
    if not plotted_any:
        plt.close(fig)
        warn_skip('No reusable training-history artifacts were found for the sweep comparison cell.')
    else:
        axes[0].set_title('Validation Total Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].legend(fontsize=8)
        axes[1].set_title('Validation Reconstruction Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].legend(fontsize=8)
        fig.savefig(PLOTS_DIR / 'latent_dim_sweep_training_curves.png', dpi=160, bbox_inches='tight')
        display(fig)
        plt.close(fig)


### Inspect The Best Beta Run

This cell identifies the best saved beta run by validation-threshold F1 and displays its saved evaluation summary so the sweep still ends with a concrete recommended configuration.


In [ ]:
if not sweep_ready:
    warn_skip('Best-run inspection is unavailable because the saved sweep summary is missing.')
else:
    best_row = latent_dim_sweep_df.iloc[0].to_dict()
    best_value = best_row['latent_dim']
    best_tag = latent_dim_tag(best_value)
    best_run_dir = SWEEP_ROOT / best_tag
    best_eval_summary_path = best_run_dir / 'results' / 'evaluation' / 'summary.json'
    best_test_scores_path = best_run_dir / 'results' / 'evaluation' / 'test_scores.csv'
    best_threshold_sweep_path = best_run_dir / 'results' / 'evaluation' / 'threshold_sweep.csv'
    if not all((path.exists() for path in [best_eval_summary_path, best_test_scores_path, best_threshold_sweep_path])):
        warn_skip(f'Best-run evaluation artifacts are missing for {best_tag}. Skipping this cell.')
    else:
        best_eval_summary = json.loads(best_eval_summary_path.read_text(encoding='utf-8'))
        best_test_scores_df = pd.read_csv(best_test_scores_path)
        best_threshold_sweep_df = pd.read_csv(best_threshold_sweep_path)
        cm = best_eval_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])
        cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
        print(f'Best latent-dim by validation-threshold F1: {best_value}')
        print(f'Best run directory: {best_run_dir}')
        display(pd.DataFrame([best_row]))
        display(pd.DataFrame([best_eval_summary['metrics_at_validation_threshold']]))
        display(pd.DataFrame([best_eval_summary['best_threshold_sweep']]))
        display(cm_df)
